In [ ]:
import modal
app = modal.App("resnet18-transfer-learning")

data_vol = modal.Volume.from_name("plant-pictures", create_if_missing=True)
ckpt_vol = modal.Volume.from_name("plant-models", create_if_missing=True)

image = (
    modal.Image.debian_slim()
    .pip_install(
        "torch",
        "torchvision",
        "pandas",
        "scikit-learn",
        "pillow",
    )
    .add_local_file(
        "dataset_and_CNN_utils.py",
        remote_path="/root/dataset_and_CNN_utils.py",
    )
)

## 2. Training function

In [27]:
@app.function(
    image=image,
    volumes={"/data": data_vol, "/ckpt": ckpt_vol},
    timeout=10800,
    gpu="T4",
)
def train_resnet18_transfer(
    num_epochs=5,
    batch_size=64,
    lr=1e-5,
    weight_decay=1e-4,
    image_size=224,
    num_workers=4,
    prefetch_factor=2,
    threshold=0.5,
):
    import json
    from pathlib import Path

    import pandas as pd
    import torch
    import torch.nn as nn
    from torchvision.models import resnet18, ResNet18_Weights

    from dataset_and_CNN_utils import make_data_loader

    if num_workers <= 0:
        raise ValueError(
            "Use num_workers >= 1 with your current make_data_loader(), "
            "because it passes prefetch_factor to DataLoader."
        )

    try:
        data_vol.reload()
        ckpt_vol.reload()
    except Exception:
        pass

    torch.backends.cudnn.benchmark = True

    data_root = Path("/data/data")
    image_dir = data_root / "train_images"
    train_csv = data_root / "train_split.csv"
    val_csv = data_root / "val_split.csv"
    classes_path = data_root / "classes.json"

    with open(classes_path, "r") as f:
        classes = json.load(f)

    num_classes = len(classes)

    print("Starting ResNet-18 transfer learning.", flush=True)
    print(f"Number of classes: {num_classes}", flush=True)
    print(f"Classes: {classes}", flush=True)
    print(f"Image size: {image_size}", flush=True)
    print(f"Batch size: {batch_size}", flush=True)
    print(f"Learning rate: {lr}", flush=True)
    print(f"Workers: {num_workers}", flush=True)

    train_loader = make_data_loader(
        csv_file=train_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        shuffle=True,
        prefetch_factor=prefetch_factor,
    )

    val_loader = make_data_loader(
        csv_file=val_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        shuffle=False,
        prefetch_factor=prefetch_factor,
    )

    print(f"Train batches: {len(train_loader)}", flush=True)
    print(f"Validation batches: {len(val_loader)}", flush=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}", flush=True)

    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # Unfreeze all weights.
    for param in model.parameters():
        param.requires_grad = True

    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    def compute_multilabel_f1_from_logits(logits, targets, threshold=0.5, eps=1e-8):
        preds = (logits >= threshold).float()
        targets = targets.float()

        tp_micro = (preds * targets).sum()
        fp_micro = (preds * (1 - targets)).sum()
        fn_micro = ((1 - preds) * targets).sum()

        micro_f1 = (2 * tp_micro) / (
            2 * tp_micro + fp_micro + fn_micro + eps
        )

        tp_class = (preds * targets).sum(dim=0)
        fp_class = (preds * (1 - targets)).sum(dim=0)
        fn_class = ((1 - preds) * targets).sum(dim=0)

        macro_f1_per_class = (2 * tp_class) / (
            2 * tp_class + fp_class + fn_class + eps
        )

        macro_f1 = macro_f1_per_class.mean()

        return micro_f1.item(), macro_f1.item()

    ckpt_dir = Path("/ckpt")
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_model_path = ckpt_dir / "best_resnet18_transfer.pt"
    last_model_path = ckpt_dir / "last_resnet18_transfer.pt"
    history_path = ckpt_dir / "resnet18_transfer_history.csv"

    history = []
    best_val_macro_f1 = -1.0

    for epoch in range(num_epochs):
        print(f"Starting epoch {epoch + 1}/{num_epochs}", flush=True)

        model.train()

        train_loss_sum = 0.0
        train_num_samples = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_size_actual = images.size(0)
            train_loss_sum += loss.item() * batch_size_actual
            train_num_samples += batch_size_actual

            if batch_idx % 25 == 0:
                print(
                    f"Epoch {epoch + 1}/{num_epochs} | "
                    f"Batch {batch_idx}/{len(train_loader)} | "
                    f"train_loss={loss.item():.4f}",
                    flush=True,
                )

        train_loss = train_loss_sum / train_num_samples

        model.eval()

        val_loss_sum = 0.0
        val_num_samples = 0

        all_val_logits = []
        all_val_targets = []

        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    logits = model(images)
                    loss = criterion(logits, targets)

                batch_size_actual = images.size(0)
                val_loss_sum += loss.item() * batch_size_actual
                val_num_samples += batch_size_actual

                all_val_logits.append(logits.detach().cpu())
                all_val_targets.append(targets.detach().cpu())

        val_loss = val_loss_sum / val_num_samples

        all_val_logits = torch.cat(all_val_logits, dim=0)
        all_val_targets = torch.cat(all_val_targets, dim=0)

        val_micro_f1, val_macro_f1 = compute_multilabel_f1_from_logits(
            logits=all_val_logits,
            targets=all_val_targets,
            threshold=threshold,
        )

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1,
            "lr": lr,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "image_size": image_size,
            "threshold": threshold,
        }

        history.append(row)

        print(
            f"Epoch {epoch + 1}/{num_epochs} finished | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_micro_f1={val_micro_f1:.4f} | "
            f"val_macro_f1={val_macro_f1:.4f}",
            flush=True,
        )

        pd.DataFrame(history).to_csv(history_path, index=False)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "classes": classes,
                    "num_classes": num_classes,
                    "epoch": epoch + 1,
                    "val_loss": val_loss,
                    "val_micro_f1": val_micro_f1,
                    "val_macro_f1": val_macro_f1,
                    "image_size": image_size,
                    "threshold": threshold,
                    "architecture": "resnet18",
                    "pretrained": True,
                    "all_weights_unfrozen": True,
                },
                best_model_path,
            )

            print(f"Saved new best model to {best_model_path}", flush=True)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "num_classes": num_classes,
            "epoch": num_epochs,
            "image_size": image_size,
            "threshold": threshold,
            "architecture": "resnet18",
            "pretrained": True,
            "all_weights_unfrozen": True,
        },
        last_model_path,
    )

    pd.DataFrame(history).to_csv(history_path, index=False)

    try:
        ckpt_vol.commit()
    except Exception:
        pass

    return {
        "num_epochs": num_epochs,
        "best_val_macro_f1": best_val_macro_f1,
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "history_path": str(history_path),
        "num_classes": num_classes,
        "classes": classes,
    }

In [28]:
@app.function(image=image, volumes={"/data": data_vol, "/ckpt": ckpt_vol})
def prepare_data():
    import json
    from pathlib import Path

    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MultiLabelBinarizer

    try:
        data_vol.reload()
    except Exception:
        pass

    data_root = Path("/data/data")
    train_csv = data_root / "train.csv"
    image_dir = data_root / "train_images"

    df = pd.read_csv(train_csv)

    df["label_list"] = df["labels"].apply(lambda x: str(x).split())

    mlb = MultiLabelBinarizer()
    mlb.fit(df["label_list"])

    train_df, val_df = train_test_split(
        df,
        test_size=0.15,
        random_state=42,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(data_root / "train_split.csv", index=False)
    val_df.to_csv(data_root / "val_split.csv", index=False)

    classes = list(mlb.classes_)

    with open(data_root / "classes.json", "w") as f:
        json.dump(classes, f, indent=2)

    meta = {
        "num_train": int(len(train_df)),
        "num_val": int(len(val_df)),
        "classes": classes,
        "num_classes": int(len(classes)),
    }

    with open(data_root / "dataset_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    try:
        data_vol.commit()
    except Exception:
        pass

    return meta


In [29]:
with app.run():
    results = train_resnet18_transfer.remote(
        num_epochs=10,
        batch_size=64,
        lr=1e-5,
        weight_decay=1e-5,
        image_size=224,
        num_workers=4,
        prefetch_factor=2,
        threshold=0.5,
    )

    print(results)

{'num_epochs': 10, 'best_val_macro_f1': 0.8635921478271484, 'best_model_path': '/ckpt/best_resnet18_transfer.pt', 'last_model_path': '/ckpt/last_resnet18_transfer.pt', 'history_path': '/ckpt/resnet18_transfer_history.csv', 'num_classes': 6, 'classes': ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab']}
